# Lab 10: Seeing Colours That Aren't There

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 1, *The Reality Generator*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-10.ipynb)

**What you will do:** rate how strong an illusory colour wash looks with and without its inducing arcs, compare your ratings with the book's claim that hiding the arcs removes it, and test whether a classical edge detector "sees" the same boundary you do.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 10 you looked at a neon-spreading (Ehrenstein) pattern of black lines with a few short
coloured arcs, and saw a faint wash of that colour seem to fill the enclosed white space between
them -- a wash that vanished the instant you covered the coloured arcs.

## Record your results

Rate the strength of the illusory colour wash (0 = no colour at all, 10 = as vivid as a real
coloured disc) with the coloured arcs visible, and again after covering them (Step 2), for the
printed figure or your own DIY grid (Step 3).

In [ ]:
# example data: replace with your own ratings
neon_log = pd.DataFrame({
    "condition": ["arcs visible", "arcs covered"],
    "wash_rating_0to10": [6, 0],   # example data
})
display(neon_log)
neon_log.plot.bar(x="condition", y="wash_rating_0to10", legend=False,
                   color=["mediumblue", "grey"], figsize=(5, 3))
plt.ylabel("illusory colour-wash rating"); plt.ylim(0, 10); plt.tight_layout(); plt.show()

## Compare

Lab 10 states: "The moment you hide them [the coloured arcs], the illusory colour wash
disappears, and you see only white paper with black lines."

In [ ]:
visible = neon_log.loc[neon_log.condition == "arcs visible", "wash_rating_0to10"].iloc[0]
covered = neon_log.loc[neon_log.condition == "arcs covered", "wash_rating_0to10"].iloc[0]
print(f"Rating with arcs visible: {visible}/10, with arcs covered: {covered}/10")
if covered == 0 and visible > 0:
    print("Matches the book: covering the arcs removes the illusion completely.")
elif covered < visible:
    print("Mostly matches the book: the rating dropped when the arcs were covered, even if not")
    print("all the way to zero -- a faint afterimage of the wash can briefly linger.")
else:
    print("Does not match the book's expected pattern -- try covering the arcs completely,")
    print("including a small margin around them, since a visible sliver of colour is enough to")
    print("sustain the illusory boundary.")

## The AI side

The book notes that "edges alone don't define objects" for a computer vision system: a
boundary-detection algorithm like the classic Canny edge detector marks only where brightness
changes sharply, with no separate step that "fills in" an enclosed surface the way your visual
cortex does. Below, we build a version of the neon-spreading figure and run Canny edge detection
on it, to see whether a classical edge detector finds any boundary at all where you perceive a
glowing coloured disc.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import cv2
from PIL import Image, ImageDraw

size = 300
img = Image.new("RGB", (size, size), "white")
draw = ImageDraw.Draw(img)
centre = size / 2
n_lines = 16
for i in range(n_lines):
    angle = 2 * np.pi * i / n_lines
    dx, dy = np.cos(angle), np.sin(angle)
    inner = (centre + dx * 40, centre + dy * 40)
    outer = (centre + dx * 130, centre + dy * 130)
    mid = (centre + dx * 65, centre + dy * 65)
    draw.line([mid, outer], fill="black", width=3)
    draw.line([inner, mid], fill=(30, 80, 220), width=3)   # the coloured segment that induces the illusion

grey = np.array(img.convert("L"))
edges = cv2.Canny(grey, 80, 160)

# Sample the edge map on the ring where a human sees the illusory circular boundary
theta = np.linspace(0, 2 * np.pi, 200)
ring_x = (centre + 65 * np.cos(theta)).astype(int).clip(0, size - 1)
ring_y = (centre + 65 * np.sin(theta)).astype(int).clip(0, size - 1)
edge_hits = edges[ring_y, ring_x]

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img); axes[0].set_title("Neon-spreading figure", fontsize=9)
axes[1].imshow(edges, cmap="gray"); axes[1].set_title("Canny edges", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Pixels along the illusory circle that Canny marks as an edge: {edge_hits.mean():.0%}")
print("(the coloured line segments themselves are marked; the gaps between them, where you see")
print("a continuous glowing boundary, are not).")

Canny finds exactly what is physically there: short marked segments, with nothing in the gaps.
It does not hallucinate the connecting arc the way your visual cortex does. This is the opposite
kind of AI/brain mismatch from most of this chapter's other demonstrations: instead of an AI
system sharing your hallucination, the classical algorithm here is more literal than you are.
Modern learned segmentation systems (Segment Anything and similar) close this gap by learning,
from millions of examples, when a broken boundary should be treated as continuous -- closer to
what your brain does, but for a learned statistical reason, not because it "sees" a real disc.

## Questions to think about

1. Your wash_rating dropped (or should have dropped) to near zero when the arcs were covered. What does this tell you about whether the "surface completion" system needs any local edge evidence to work with, or whether it can invent a coloured surface from nothing at all?
2. In the Canny demo, the edge detector marks the coloured segments but not the gaps between them. Using the book's two-stage model (boundary detection, then surface filling), which of the two stages does Canny alone perform, and which stage is entirely missing from that code?
3. The book connects neon spreading to stereotyping, filling in a whole "surface" of assumed attributes from a few visible cues. Using your own wash_rating results, what might vary between people about how strong that filling-in is -- and could that same variation apply to how readily different people fill in social stereotypes from minimal cues?
4. Sasaki and Watanabe (2004) found that the illusory colour itself activates the primary visual cortex (V1), the earliest stage of vision, not just higher-level "interpretation" areas. Why might that finding matter for whether we should call neon spreading a "low-level" visual effect or a "high-level" cognitive judgement?

## Challenge

Vary the DIY grid from Step 3: draw a second version where the coloured segments are spaced much
closer to the centre (making the implied disc smaller) and a third where they are spaced much
farther out (making the implied disc larger). Rate the vividness of the colour wash for all three
sizes.

In [ ]:
# Example data: replace with your own ratings for grids of different sizes.
size_test = pd.DataFrame({
    "implied_disc_radius_cm": [1.5, 3.0, 5.0],   # example data: how far out you drew the coloured segments
    "wash_rating_0to10": [7, 6, 3],                # example data
})
display(size_test)
size_test.plot.scatter(x="implied_disc_radius_cm", y="wash_rating_0to10", figsize=(5, 3.5))
plt.ylim(0, 10); plt.tight_layout(); plt.show()
print("If the rating falls as the implied disc gets larger, that is consistent with the general")
print("idea that neon spreading weakens as the gap the visual system has to 'fill' grows -- though")
print("the book does not give a specific size-vividness relationship, so treat any pattern you")
print("find as your own exploratory result, not a confirmed effect.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-10.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")